# 03 — Exposure Labels v2

Identifies anchor posts and classifies users as exposed/unexposed.

**Anchor post definition:** keyword filter + at least `MIN_DIMS_FOR_ANCHOR` SVM dimensions above upper tertile (p67)
**Exposed user:** commented on anchor thread; assigned `exposure_intensity` (max n_dims) and `exposure_prob` (popularity-weighted)
**Unexposed user:** active same week as anchor events, never commented on anchor thread

**Outputs:** `anchor_posts_v2.parquet`, `exposure_labels_v2.parquet`


In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║                     STUDY CONFIG                            ║
# ║  Change only this cell to run on a different dataset        ║
# ╚══════════════════════════════════════════════════════════════╝

STUDY_ID   = 'gradadmissions'
SUBREDDIT  = 'gradadmissions'

# Input files (relative to ROOT)
POSTS_CLEAN_FILE  = 'cleaned_output/r_gradadmissions_posts.cleaned.jsonl'
COMMENTS_RAW_FILE = 'Grad Admissions Comments.jsonl'

# Anchor period cycles — add entries for more cycles
CYCLES = {
    1: {'anchor_start': '2023-09-01', 'anchor_end': '2023-11-30',
        'active_start': '2023-08-01', 'active_end':  '2024-05-31'},
    2: {'anchor_start': '2024-09-01', 'anchor_end': '2024-11-30',
        'active_start': '2024-08-01', 'active_end':  '2025-05-31'},
}

# Keyword filter — edit for a different community
NEGATIVE_KEYWORDS = [
    r'\breject(?:ed|ion)\b',     r'\bdeclin(?:ed|ing)\b',
    r'\bwaitlist(?:ed)?\b',       r'\bfunding\s+(?:lost|cut|removed|denied|gap|issue)\b',
    r'\bno\s+funding\b',          r'\bstipend\b',
    r'\bwithdrew?\s+(?:offer|admission)\b', r'\bacceptance\s+rate\b',
    r'\bno\s+(?:offer|response|interview)\b', r'\bsilence\s+from\b',
    r'\bnot\s+(?:accepted|admitted|selected)\b', r'\bgave\s+up\b',
    r'\bmental\s+health\b',        r'\banxi(?:ous|ety)\b',
    r'\bdepress(?:ed|ing|ion)\b',  r'\bstress(?:ed|ful)?\b',
    r'\boverwhelm(?:ed|ing)\b',    r'\bscared\b',
    r'\bworr(?:ied|ying)\b',       r'\bfalling\s+apart\b',
    r'\bbreaking\s+down\b',        r"\bcan(?:'t|not)\s+(?:take|handle|cope)\b",
    r'\bno\s+chance\b',            r'\bnot\s+good\s+enough\b',
    r'\bregret\b', r'\bfailed\b',  r'\bimposter\b',
]

# Threshold percentile per dimension (2/3 = upper tertile p67)
TERTILE_QUANTILE    = 2 / 3
# Min dimensions above threshold for a post to qualify as anchor
MIN_DIMS_FOR_ANCHOR = 1   # 1=OR, 2=majority, 3=AND

# Popularity weighting: 1.0=full log-normalised upvote score, 0.0=uniform
POPULARITY_WEIGHT   = 1.0


In [2]:
import json, re
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

ROOT      = Path('..').resolve()
DATA_DIR  = ROOT / 'data' / 'processed_v2'
MODEL_DIR = ROOT / 'models'
POSTS_PATH    = ROOT / POSTS_CLEAN_FILE
COMMENTS_PATH = ROOT / COMMENTS_RAW_FILE

keyword_pattern = re.compile('|'.join(NEGATIVE_KEYWORDS), re.IGNORECASE)

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line: rows.append(json.loads(line))
    return rows

print(f'Study: {STUDY_ID} | r/{SUBREDDIT}')
print(f'Cycles: {list(CYCLES.keys())}')
print(f'Anchor threshold: p{TERTILE_QUANTILE*100:.0f}, min_dims={MIN_DIMS_FOR_ANCHOR}')


Study: gradadmissions | r/gradadmissions
Cycles: [1, 2]
Anchor threshold: p67, min_dims=1


## 1) Load raw posts


In [3]:
raw_posts = load_jsonl(POSTS_PATH)
posts = pd.DataFrame([{
    'id':           r['record_id'],
    'author':       r['author'],
    'created_dt':   pd.Timestamp(r['created_utc'], unit='s', tz='UTC'),
    'clean_text':   r.get('clean_text', ''),
    'score':        r.get('score', 0),
    'num_comments': r.get('num_comments', 0),
} for r in raw_posts])
print(f'Posts loaded: {len(posts):,} from {posts["author"].nunique():,} authors')
print(f'Date range: {posts["created_dt"].min().date()} → {posts["created_dt"].max().date()}')


Posts loaded: 88,441 from 45,309 authors
Date range: 2023-08-01 → 2025-07-30


## 2) Filter to anchor periods and score with SVM classifiers


In [4]:
def assign_cycle(dt):
    for cycle, w in CYCLES.items():
        if pd.Timestamp(w['anchor_start'], tz='UTC') <= dt <= pd.Timestamp(w['anchor_end'] + ' 23:59:59', tz='UTC'):
            return cycle
    return None

posts['cycle'] = posts['created_dt'].apply(assign_cycle)
anchor_candidates = posts[posts['cycle'].notna()].copy()
print(f'Posts in anchor periods: {len(anchor_candidates):,}')
print(anchor_candidates['cycle'].value_counts().sort_index())


Posts in anchor periods: 15,973
cycle
1.0    7786
2.0    8187
Name: count, dtype: int64


In [5]:
clf_anx = joblib.load(MODEL_DIR / 'clf_anxiety.joblib')
clf_dep = joblib.load(MODEL_DIR / 'clf_depression.joblib')
clf_str = joblib.load(MODEL_DIR / 'clf_stress.joblib')
print('Classifiers loaded.')

def sigmoid(x): return 1 / (1 + np.exp(-x))

texts = anchor_candidates['clean_text'].tolist()
anchor_candidates['anx_score'] = sigmoid(clf_anx.decision_function(texts))
anchor_candidates['dep_score'] = sigmoid(clf_dep.decision_function(texts))
anchor_candidates['str_score'] = sigmoid(clf_str.decision_function(texts))
anchor_candidates['mean_mh_score'] = anchor_candidates[['anx_score','dep_score','str_score']].mean(axis=1)
print(f'Scored {len(anchor_candidates):,} anchor-period posts')


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.8.0 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.8.0 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LinearSVC from version 1.8.0 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.8.0 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Classifiers loaded.


Scored 15,973 anchor-period posts


## 3) Apply keyword filter + per-dimension tertile threshold → anchor posts


In [6]:
anchor_candidates['has_neg_keyword'] = anchor_candidates['clean_text'].str.contains(
    keyword_pattern, na=False
)
kw_filtered = anchor_candidates[anchor_candidates['has_neg_keyword']].copy()
print(f'After keyword filter: {len(kw_filtered):,}')

DIMS = ['anx_score', 'dep_score', 'str_score']
tertile_thresholds = {dim: kw_filtered[dim].quantile(TERTILE_QUANTILE) for dim in DIMS}
print(f'\nPer-dimension thresholds (p{TERTILE_QUANTILE*100:.0f}):')
for dim, thr in tertile_thresholds.items():
    print(f'  {dim}: {thr:.4f}')

for dim in DIMS:
    kw_filtered[f'{dim}_above'] = kw_filtered[dim] > tertile_thresholds[dim]
kw_filtered['n_dims'] = (kw_filtered['anx_score_above'].astype(int) +
                         kw_filtered['dep_score_above'].astype(int) +
                         kw_filtered['str_score_above'].astype(int))

anchor_posts = kw_filtered[kw_filtered['n_dims'] >= MIN_DIMS_FOR_ANCHOR].copy()
print(f'\nAnchor posts (n_dims >= {MIN_DIMS_FOR_ANCHOR}): {len(anchor_posts):,}')
print(anchor_posts['cycle'].value_counts().sort_index())
print('\nn_dims distribution:')
print(anchor_posts['n_dims'].value_counts().sort_index().rename('posts'))


After keyword filter: 2,237

Per-dimension thresholds (p67):
  anx_score: 0.4384
  dep_score: 0.4066
  str_score: 0.4703

Anchor posts (n_dims >= 1): 1,024
cycle
1.0    458
2.0    566
Name: count, dtype: int64

n_dims distribution:
n_dims
1    309
2    216
3    499
Name: posts, dtype: int64


In [7]:
anchor_posts[['id','author','created_dt','cycle','clean_text',
              'anx_score','dep_score','str_score','mean_mh_score',
              'n_dims','score','num_comments']]\
    .to_parquet(DATA_DIR / 'anchor_posts_v2.parquet', index=False)
print('Saved anchor_posts_v2.parquet')

anchor_ids_by_cycle = {c: set(anchor_posts[anchor_posts['cycle']==c]['id']) for c in CYCLES}
anchor_authors_by_cycle = {c: set(anchor_posts[anchor_posts['cycle']==c]['author']) for c in CYCLES}
post_ndims = anchor_posts.set_index('id')['n_dims'].to_dict()
print(f'Anchor IDs — ' + ' | '.join(f'C{c}: {len(v):,}' for c, v in anchor_ids_by_cycle.items()))


Saved anchor_posts_v2.parquet
Anchor IDs — C1: 458 | C2: 566


## 4) Load comments → identify exposed users


In [8]:
import datetime
print('Loading comments...')
comment_rows = []
with open(COMMENTS_PATH) as f:
    for line in f:
        r = json.loads(line)
        comment_rows.append({
            'id':         r.get('id',''),
            'author':     r.get('author',''),
            'post_id':    r.get('link_id','').replace('t3_',''),
            'created_dt': pd.Timestamp(r['created_utc'], unit='s', tz='UTC'),
        })
comments = pd.DataFrame(comment_rows)
print(f'Comments loaded: {len(comments):,} from {comments["author"].nunique():,} authors')


Loading comments...


Comments loaded: 500,688 from 80,466 authors


In [9]:
all_anchor_ids = set().union(*anchor_ids_by_cycle.values())
anchor_comments = comments[comments['post_id'].isin(all_anchor_ids)].copy()
print(f'Comments on anchor posts: {len(anchor_comments):,}')

def comment_cycle(post_id):
    for c, ids in anchor_ids_by_cycle.items():
        if post_id in ids: return c
    return None

anchor_comments['cycle'] = anchor_comments['post_id'].apply(comment_cycle)
print(anchor_comments['cycle'].value_counts().sort_index())


Comments on anchor posts: 7,761
cycle
1    3074
2    4687
Name: count, dtype: int64


In [10]:
global_max_log_score = np.log1p(anchor_posts['score'].clip(lower=0).max())
exposed_records = []

for cycle in CYCLES:
    cycle_comments = anchor_comments[anchor_comments['cycle'] == cycle]
    excluded = anchor_authors_by_cycle[cycle]
    eligible = cycle_comments[~cycle_comments['author'].isin(excluded)].copy()

    ap_scores = anchor_posts[['id','score','n_dims']].rename(
        columns={'score':'post_upvote_score','n_dims':'post_ndims'})
    eligible = eligible.merge(ap_scores, left_on='post_id', right_on='id', how='left')
    eligible['post_ndims']        = eligible['post_ndims'].fillna(1).astype(int)
    eligible['post_upvote_score'] = eligible['post_upvote_score'].fillna(0).clip(lower=0)

    author_intensity  = eligible.groupby('author')['post_ndims'].max()
    author_max_score  = eligible.groupby('author')['post_upvote_score'].max()

    for author in author_intensity.index:
        if POPULARITY_WEIGHT > 0 and global_max_log_score > 0:
            p_pop = np.log1p(author_max_score[author]) / global_max_log_score
        else:
            p_pop = 1.0
        exposed_records.append({
            'author': author, 'exposed': True, 'cycle': cycle,
            'exposure_intensity': int(author_intensity[author]),
            'exposure_prob':      float(p_pop),
        })
    print(f'Cycle {cycle} — exposed: {len(author_intensity):,} '
          f'(excluded {len(set(cycle_comments["author"]) & excluded):,} anchor authors)')
    print(f'  Intensity: {author_intensity.value_counts().sort_index().to_dict()}')

exposed_df = pd.DataFrame(exposed_records)
print(f'\nTotal exposed: {len(exposed_df):,}')


Cycle 1 — exposed: 1,189 (excluded 193 anchor authors)
  Intensity: {1: 268, 2: 156, 3: 765}
Cycle 2 — exposed: 1,682 (excluded 277 anchor authors)
  Intensity: {1: 377, 2: 287, 3: 1018}

Total exposed: 2,871


## 5) Identify unexposed users


In [11]:
def iso_week(dt): return dt.strftime('%G-W%V')
posts['iso_week']    = posts['created_dt'].apply(iso_week)
comments['iso_week'] = comments['created_dt'].apply(iso_week)
anchor_posts['iso_week'] = anchor_posts['created_dt'].apply(iso_week)

anchor_weeks_by_cycle = {c: set(anchor_posts[anchor_posts['cycle']==c]['iso_week']) for c in CYCLES}

unexposed_records = []
for cycle in CYCLES:
    anchor_weeks = anchor_weeks_by_cycle[cycle]
    exposed_this_cycle = set(exposed_df[exposed_df['cycle']==cycle]['author'])
    active = set(posts[posts['iso_week'].isin(anchor_weeks)]['author']) | \
             set(comments[comments['iso_week'].isin(anchor_weeks)]['author'])
    unexposed = active - exposed_this_cycle
    for author in unexposed:
        unexposed_records.append({'author': author, 'exposed': False, 'cycle': cycle,
                                  'exposure_intensity': 0, 'exposure_prob': 0.0})
    print(f'Cycle {cycle} — active: {len(active):,} | exposed: {len(exposed_this_cycle):,} | unexposed: {len(unexposed):,}')

unexposed_df = pd.DataFrame(unexposed_records)
print(f'\nTotal unexposed: {len(unexposed_df):,}')


Cycle 1 — active: 10,326 | exposed: 1,189 | unexposed: 9,226
Cycle 2 — active: 12,915 | exposed: 1,682 | unexposed: 11,295

Total unexposed: 20,521


## 6) Combine and save


In [12]:
exposure_df = pd.concat([exposed_df, unexposed_df], ignore_index=True)
exposure_df['exposure_intensity'] = exposure_df['exposure_intensity'].fillna(0).astype(int)
exposure_df['exposure_prob']      = exposure_df['exposure_prob'].fillna(0.0).astype(float)

print(f'Total records: {len(exposure_df):,} | Unique users: {exposure_df["author"].nunique():,}')
print('\nExposed vs unexposed by cycle:')
print(exposure_df.groupby(['cycle','exposed']).size().unstack(fill_value=0))
print('\nExposure intensity (exposed only):')
print(exposure_df[exposure_df['exposed']].groupby(['cycle','exposure_intensity'])['author'].count().unstack(fill_value=0))
print('\nExposure prob summary (exposed only):')
print(exposure_df[exposure_df['exposed']].groupby('cycle')['exposure_prob'].describe().round(3))

exposure_df.to_parquet(DATA_DIR / 'exposure_labels_v2.parquet', index=False)
print('\nSaved exposure_labels_v2.parquet')
print('Columns:', exposure_df.columns.tolist())


Total records: 23,392 | Unique users: 22,518

Exposed vs unexposed by cycle:
exposed  False  True 
cycle                
1         9226   1189
2        11295   1682

Exposure intensity (exposed only):
exposure_intensity    1    2     3
cycle                             
1                   268  156   765
2                   377  287  1018

Exposure prob summary (exposed only):
        count   mean    std  min    25%    50%    75%    max
cycle                                                       
1      1189.0  0.484  0.246  0.0  0.270  0.518  0.663  0.899
2      1682.0  0.591  0.300  0.0  0.387  0.636  0.864  1.000

Saved exposure_labels_v2.parquet
Columns: ['author', 'exposed', 'cycle', 'exposure_intensity', 'exposure_prob']


## 7) Sanity checks


In [13]:
both_cycles = exposure_df.groupby('author')['cycle'].nunique()
print(f'Users in both cycles: {(both_cycles==2).sum():,}')
exposed_both = exposure_df[exposure_df['exposed']].groupby('author')['cycle'].nunique()
print(f'Exposed in both cycles: {(exposed_both==2).sum():,}')
print(f'\nexposure_prob range (exposed): {exposure_df[exposure_df["exposed"]]["exposure_prob"].agg(["min","max"]).round(3).to_dict()}')
print(f'Average exposure_prob among exposed: {exposure_df[exposure_df["exposed"]]["exposure_prob"].mean():.4f}')


Users in both cycles: 874
Exposed in both cycles: 67

exposure_prob range (exposed): {'min': 0.0, 'max': 1.0}
Average exposure_prob among exposed: 0.5466
